# CFLOW Two-Stage on Severstal (Protocol-Aligned)

This notebook keeps a **two-stage** setup: Stage-1 CFLOW anomaly screening + Stage-2 ResNet known-class classifier.

It matches the one-stage operating-point protocol with FPR targets `[0.05, 0.10, 0.15, 0.20]` and uses **validation-normal only** for stage-1 threshold calibration.

In [3]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir", "--force-reinstall",
    "numpy==1.26.4"
])

0

In [4]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-c", "import numpy; print(numpy.__version__, numpy.__file__)"],
    capture_output=True, text=True
)
print(result.stdout)  # should show 1.26.4 and a path under /usr/local/lib

1.26.4 /usr/local/lib/python3.12/dist-packages/numpy/__init__.py



In [1]:
# Must be first cell after restart — before any other imports
import numpy as np
print("numpy:", np.__version__)          # must be 1.26.4 before continuing

import torch, torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

numpy: 1.26.4
torch: 2.10.0+cu128
torchvision: 0.25.0+cu128


In [2]:
import numpy as np, torch, torchvision

print(f"numpy:       {np.__version__}")
print(f"torch:       {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")

# numpy ABI must be 1.x
assert np.__version__.startswith('1.26'), \
    f"numpy must be 1.26.x, got {np.__version__}"

# CUDA must be available
assert torch.cuda.is_available(), "No CUDA — check runtime type (Runtime > Change runtime type > GPU)"

# Quick functional smoke test
x = torch.randn(2, 3, 32, 32).cuda()
assert x.shape == (2, 3, 32, 32)

print("Preflight OK")

numpy:       1.26.4
torch:       2.10.0+cu128
torchvision: 0.25.0+cu128
Preflight OK


In [19]:
# 1) Global config
from pathlib import Path

PILOT_MODE = False  # set False for heavier full run
SEED = 42
FPR_GRID = [0.05, 0.10, 0.15, 0.20]
assert 0.15 in FPR_GRID

DRIVE_ROOT = Path('/content/drive/MyDrive')
DATA_ROOT = DRIVE_ROOT / 'datasets' / 'severstal'
OUT_ROOT = DRIVE_ROOT / 'fyp_outputs' / 'severstral_cflow_two_stage_full'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

FYP_REPO = Path('/content/FYP-code')
CFLOW_REPO = Path('/content/cflow-ad')

# runtime class must be an official MVTec class in cflow-ad
CFLOW_RUNTIME_CLASS = 'bottle'

# split labels from severstal-osr config
SPLITS = ['a', 'b', 'c', 'd']

# stage-2 unknown threshold (known-val confidence lower-tail)
STAGE2_UNKNOWN_Q = 0.10

print('PILOT_MODE:', PILOT_MODE)
print('FPR grid:', FPR_GRID)
print('OUT_ROOT:', OUT_ROOT)


PILOT_MODE: False
FPR grid: [0.05, 0.1, 0.15, 0.2]
OUT_ROOT: /content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full


In [20]:
# 2) Setup repos, deps, and paths
import os, sys, subprocess


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.check_call(cmd, cwd=cwd)

if not FYP_REPO.exists():
    run(['git', 'clone', 'https://github.com/spinelessknave8/FYP_code.git', str(FYP_REPO)])
else:
    run(['git', 'pull', '--ff-only'], cwd=str(FYP_REPO))

if not CFLOW_REPO.exists():
    run(['git', 'clone', 'https://github.com/gudovskiy/cflow-ad.git', str(CFLOW_REPO)])
else:
    run(['git', 'fetch', '--all', '--prune'], cwd=str(CFLOW_REPO))
    run(['git', 'checkout', 'master'], cwd=str(CFLOW_REPO))
    run(['git', 'reset', '--hard', 'origin/master'], cwd=str(CFLOW_REPO))

run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==0.9.7', 'FrEIA==0.2', 'scikit-learn==1.3.2'])

os.chdir(str(FYP_REPO))
if str(FYP_REPO / 'severstral-osr' / 'src') not in sys.path:
    sys.path.insert(0, str(FYP_REPO / 'severstral-osr' / 'src'))

print('Setup complete.')


+ git pull --ff-only
+ git fetch --all --prune
+ git checkout master
+ git reset --hard origin/master
+ /usr/bin/python3 -m pip install -q timm==0.9.7 FrEIA==0.2 scikit-learn==1.3.2
Setup complete.


In [21]:
# 3) Drive + dataset checks (fixed)
import json
from pathlib import Path
from google.colab import drive

# Never delete /content/drive
drive.mount('/content/drive', force_remount=True)
print("Drive mounted at /content/drive")

DATA_ROOT = Path('/content/drive/MyDrive/datasets/severstal')
required = [DATA_ROOT / 'train.csv', DATA_ROOT / 'train_images']
for p in required:
    if not p.exists():
        raise FileNotFoundError(f'Missing required dataset path: {p}')

print('Dataset root OK:', DATA_ROOT)

Mounted at /content/drive
Drive mounted at /content/drive
Dataset root OK: /content/drive/MyDrive/datasets/severstal


In [22]:
# 4) Build split manifests (a,b,c,d) with pilot/full sizing
import json, random
from collections import defaultdict
from pathlib import Path
import yaml

from data import collect_single_label_defect_samples, collect_normal_image_paths, stratified_split

rng = random.Random(SEED)
manifest_dir = OUT_ROOT / 'manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)

single = collect_single_label_defect_samples(str(DATA_ROOT), 'train.csv', 'train_images')
normal_paths_all = collect_normal_image_paths(str(DATA_ROOT), 'train.csv', 'train_images')

all_classes = ['Class_1', 'Class_2', 'Class_3', 'Class_4']

if PILOT_MODE:
    cfg_sizes = {
        'normal_train': 240,
        'normal_val': 80,
        'normal_test': 80,
        'known_train_per_class': 80,
        'known_val_per_class': 30,
        'known_test_per_class': 30,
        'unknown_val': 60,
        'unknown_test': 90,
    }
else:
    cfg_sizes = {
        'normal_train': 1200,
        'normal_val': 300,
        'normal_test': 300,
        'known_train_per_class': 300,
        'known_val_per_class': 100,
        'known_test_per_class': 100,
        'unknown_val': 300,
        'unknown_test': 300,
    }


def take_per_class(samples, per_class, seed):
    r = random.Random(seed)
    by = defaultdict(list)
    for p, c in samples:
        by[c].append((p, c))
    out = []
    for c in sorted(by):
        r.shuffle(by[c])
        out.extend(by[c][:per_class])
    return out

split_manifests = {}
for idx, split in enumerate(SPLITS):
    split_yaml = yaml.safe_load((FYP_REPO / 'severstral-osr' / 'configs' / f'split_{split}.yaml').read_text())
    known_classes = split_yaml['known_classes']
    unknown_class = [c for c in all_classes if c not in known_classes][0]

    known_all = [s for s in single if s[1] in known_classes]
    unknown_all = [s for s in single if s[1] == unknown_class]

    ktr_all, kva_all, kte_all = stratified_split(known_all, train_ratio=0.7, val_ratio=0.15, seed=SEED + idx)
    known_train = take_per_class(ktr_all, cfg_sizes['known_train_per_class'], SEED + idx)
    known_val = take_per_class(kva_all, cfg_sizes['known_val_per_class'], SEED + idx)
    known_test = take_per_class(kte_all, cfg_sizes['known_test_per_class'], SEED + idx)

    rr = random.Random(SEED + idx)
    rr.shuffle(unknown_all)
    unknown_val = unknown_all[:cfg_sizes['unknown_val']]
    unknown_test = unknown_all[cfg_sizes['unknown_val']:cfg_sizes['unknown_val'] + cfg_sizes['unknown_test']]

    normals = normal_paths_all[:]
    rr.shuffle(normals)
    ntr = normals[:cfg_sizes['normal_train']]
    nva = normals[cfg_sizes['normal_train']:cfg_sizes['normal_train'] + cfg_sizes['normal_val']]
    nts = normals[cfg_sizes['normal_train'] + cfg_sizes['normal_val']:cfg_sizes['normal_train'] + cfg_sizes['normal_val'] + cfg_sizes['normal_test']]

    manifest = {
        'split_name': f'split_{split}',
        'known_classes': known_classes,
        'unknown_class': unknown_class,
        'normal_train': ntr,
        'normal_val': nva,
        'normal_test': nts,
        'known_train': [{'path': p, 'label': c} for p, c in known_train],
        'known_val': [{'path': p, 'label': c} for p, c in known_val],
        'known_test': [{'path': p, 'label': c} for p, c in known_test],
        'unknown_val': [{'path': p, 'label': c} for p, c in unknown_val],
        'unknown_test': [{'path': p, 'label': c} for p, c in unknown_test],
    }

    outp = manifest_dir / f'split_{split}.json'
    outp.write_text(json.dumps(manifest, indent=2))
    split_manifests[split] = outp
    print(f"split_{split}:", {
        'known_classes': known_classes,
        'normal_train': len(ntr),
        'normal_val': len(nva),
        'normal_test': len(nts),
        'known_train': len(known_train),
        'known_val': len(known_val),
        'known_test': len(known_test),
        'unknown_val': len(unknown_val),
        'unknown_test': len(unknown_test),
    })


split_a: {'known_classes': ['Class_1', 'Class_2', 'Class_3'], 'normal_train': 1200, 'normal_val': 300, 'normal_test': 300, 'known_train': 736, 'known_val': 229, 'known_test': 230, 'unknown_val': 300, 'unknown_test': 216}
split_b: {'known_classes': ['Class_1', 'Class_2', 'Class_4'], 'normal_train': 1200, 'normal_val': 300, 'normal_test': 300, 'known_train': 736, 'known_val': 206, 'known_test': 208, 'unknown_val': 300, 'unknown_test': 300}
split_c: {'known_classes': ['Class_1', 'Class_3', 'Class_4'], 'normal_train': 1200, 'normal_val': 300, 'normal_test': 300, 'known_train': 900, 'known_val': 277, 'known_test': 278, 'unknown_val': 195, 'unknown_test': 0}
split_d: {'known_classes': ['Class_2', 'Class_3', 'Class_4'], 'normal_train': 1200, 'normal_val': 300, 'normal_test': 300, 'known_train': 736, 'known_val': 206, 'known_test': 208, 'unknown_val': 300, 'unknown_test': 300}


In [23]:
# 5) Patch CFLOW repo safely (reset + compatibility + score export)
import subprocess, sys
from pathlib import Path

# hard reset local edits from prior runs
subprocess.check_call(['git', 'checkout', '--', 'main.py', 'train.py', 'custom_datasets/loader.py'], cwd=str(CFLOW_REPO))

loader_py = CFLOW_REPO / 'custom_datasets' / 'loader.py'
loader_txt = loader_py.read_text()
loader_txt = loader_txt.replace('Image.ANTIALIAS', 'Image.Resampling.LANCZOS')
loader_py.write_text(loader_txt)

train_py = CFLOW_REPO / 'train.py'
train_txt = train_py.read_text()
marker = 'scores_run{c.run_name}_{c.class_name}_epoch{epoch}.npz'
if marker not in train_txt:
    anchor = '        det_roc_auc = roc_auc_score(gt_label, score_label)\n'
    inject = (
        "        det_roc_auc = roc_auc_score(gt_label, score_label)\n"
        "        np.savez(\n"
        "            os.path.join('./results', f'scores_run{c.run_name}_{c.class_name}_epoch{epoch}.npz'),\n"
        "            score_label=score_label.astype(np.float32),\n"
        "            gt_label=gt_label.astype(np.uint8),\n"
        "        )\n"
    )
    if anchor not in train_txt:
        raise RuntimeError('Could not find score anchor in train.py; patch aborted.')
    train_txt = train_txt.replace(anchor, inject, 1)
    train_py.write_text(train_txt)

# syntax guards
subprocess.check_call([sys.executable, '-m', 'py_compile', str(CFLOW_REPO / 'main.py')])
subprocess.check_call([sys.executable, '-m', 'py_compile', str(CFLOW_REPO / 'train.py')])
subprocess.check_call([sys.executable, '-m', 'py_compile', str(CFLOW_REPO / 'custom_datasets' / 'loader.py')])

print('CFLOW patching complete and compiled.')


CFLOW patching complete and compiled.


In [24]:
# Rebuild split manifests after cleanup
import json, random, sys
from collections import defaultdict
from pathlib import Path

# Paths
OUT_ROOT = Path('/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full')
DATA_ROOT = Path('/content/drive/MyDrive/datasets/severstal')  # change if your folder differs
FYP_REPO = Path('/content/FYP-code')
SEED = 42
PILOT_MODE = True
SPLITS = ['a','b','c','d']

# imports from repo
sys.path.insert(0, str(FYP_REPO / 'severstral-osr' / 'src'))
from data import collect_single_label_defect_samples, collect_normal_image_paths, stratified_split
import yaml

manifest_dir = OUT_ROOT / 'manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)

single = collect_single_label_defect_samples(str(DATA_ROOT), 'train.csv', 'train_images')
normal_paths_all = collect_normal_image_paths(str(DATA_ROOT), 'train.csv', 'train_images')
all_classes = ['Class_1','Class_2','Class_3','Class_4']

if PILOT_MODE:
    cfg_sizes = {
        'normal_train': 240, 'normal_val': 80, 'normal_test': 80,
        'known_train_per_class': 80, 'known_val_per_class': 30, 'known_test_per_class': 30,
        'unknown_val': 60, 'unknown_test': 90,
    }
else:
    cfg_sizes = {
        'normal_train': 1200, 'normal_val': 300, 'normal_test': 300,
        'known_train_per_class': 300, 'known_val_per_class': 100, 'known_test_per_class': 100,
        'unknown_val': 300, 'unknown_test': 300,
    }

def take_per_class(samples, per_class, seed):
    r = random.Random(seed)
    by = defaultdict(list)
    for p,c in samples:
        by[c].append((p,c))
    out = []
    for c in sorted(by):
        r.shuffle(by[c])
        out.extend(by[c][:per_class])
    return out

for idx, split in enumerate(SPLITS):
    split_yaml = yaml.safe_load((FYP_REPO / 'severstral-osr' / 'configs' / f'split_{split}.yaml').read_text())
    known_classes = split_yaml['known_classes']
    unknown_class = [c for c in all_classes if c not in known_classes][0]

    known_all = [s for s in single if s[1] in known_classes]
    unknown_all = [s for s in single if s[1] == unknown_class]

    ktr_all, kva_all, kte_all = stratified_split(known_all, train_ratio=0.7, val_ratio=0.15, seed=SEED+idx)
    known_train = take_per_class(ktr_all, cfg_sizes['known_train_per_class'], SEED+idx)
    known_val   = take_per_class(kva_all, cfg_sizes['known_val_per_class'], SEED+idx)
    known_test  = take_per_class(kte_all, cfg_sizes['known_test_per_class'], SEED+idx)

    rr = random.Random(SEED+idx)
    rr.shuffle(unknown_all)
    unknown_val = unknown_all[:cfg_sizes['unknown_val']]
    unknown_test = unknown_all[cfg_sizes['unknown_val']:cfg_sizes['unknown_val']+cfg_sizes['unknown_test']]

    normals = normal_paths_all[:]
    rr.shuffle(normals)
    ntr = normals[:cfg_sizes['normal_train']]
    nva = normals[cfg_sizes['normal_train']:cfg_sizes['normal_train']+cfg_sizes['normal_val']]
    nts =normals[cfg_sizes['normal_train']+cfg_sizes['normal_val']:cfg_sizes['normal_train']+cfg_sizes['normal_val']+cfg_sizes['normal_test']]

    m = {
        'split_name': f'split_{split}',
        'known_classes': known_classes,
        'unknown_class': unknown_class,
        'normal_train': ntr,
        'normal_val': nva,
        'normal_test': nts,
        'known_train': [{'path':p,'label':c} for p,c in known_train],
        'known_val':   [{'path':p,'label':c} for p,c in known_val],
        'known_test':  [{'path':p,'label':c} for p,c in known_test],
        'unknown_val': [{'path':p,'label':c} for p,c in unknown_val],
        'unknown_test':[{'path':p,'label':c} for p,c in unknown_test],
    }
    out = manifest_dir / f'split_{split}.json'
    out.write_text(json.dumps(m, indent=2))
    print("wrote:", out)

print("Manifest rebuild complete.")

wrote: /content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/manifests/split_a.json
wrote: /content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/manifests/split_b.json
wrote: /content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/manifests/split_c.json
wrote: /content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/manifests/split_d.json
Manifest rebuild complete.


In [25]:
# Rebuild CFLOW datasets robustly (after cleanup/disconnect)
import json, shutil
from pathlib import Path
from PIL import Image

build_root = OUT_ROOT / 'cflow_datasets'
build_root.mkdir(parents=True, exist_ok=True)

def to_png(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    img = Image.open(src).convert('RGB')
    img.save(dst, format='PNG')

def make_mask_like(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    img = Image.open(src)
    m = Image.new('L', img.size, color=255)
    m.save(dst, format='PNG')

def build_mvtec_layout(dst_root, normal_train, normal_test, known_def, unknown_def):
    if dst_root.exists():
        shutil.rmtree(dst_root)

    # precreate full tree
    for d in [
        dst_root / 'train' / 'good',
        dst_root / 'test' / 'good',
        dst_root / 'test' / 'known_defect',
        dst_root / 'test' / 'unknown_defect',
        dst_root / 'ground_truth' / 'known_defect',
        dst_root / 'ground_truth' / 'unknown_defect',
    ]:
        d.mkdir(parents=True, exist_ok=True)

    for i, p in enumerate(normal_train):
        to_png(p, dst_root / 'train' / 'good' / f'{i:06d}.png')

    for i, p in enumerate(normal_test):
        to_png(p, dst_root / 'test' / 'good' / f'{i:06d}.png')

    for i, d in enumerate(known_def):
        src = d['path']
        stem = f'{i:06d}'
        to_png(src, dst_root / 'test' / 'known_defect' / f'{stem}.png')
        make_mask_like(src, dst_root / 'ground_truth' / 'known_defect' / f'{stem}_mask.png')

    for i, d in enumerate(unknown_def):
        src = d['path']
        stem = f'{i:06d}'
        to_png(src, dst_root / 'test' / 'unknown_defect' / f'{stem}.png')
        make_mask_like(src, dst_root / 'ground_truth' / 'unknown_defect' / f'{stem}_mask.png')

runtime_manifest_dir = OUT_ROOT / 'runtime_manifests'
runtime_manifest_dir.mkdir(parents=True, exist_ok=True)

for split in SPLITS:
    m = json.loads((manifest_dir / f'split_{split}.json').read_text())
    split_root = build_root / f'split_{split}'
    split_root.mkdir(parents=True, exist_ok=True)

    main_root = split_root / 'main'
    val_root = split_root / 'val'

    build_mvtec_layout(main_root, m['normal_train'], m['normal_test'], m['known_test'], m['unknown_test'])
    build_mvtec_layout(val_root,  m['normal_train'], m['normal_val'],  m['known_val'],  m['unknown_val'])

    rm = {
        'split': f'split_{split}',
        'main_counts': {'normal': len(m['normal_test']), 'known': len(m['known_test']), 'unknown':
len(m['unknown_test'])},
        'val_counts':  {'normal': len(m['normal_val']),  'known': len(m['known_val']),  'unknown':
len(m['unknown_val'])},
    }
    (runtime_manifest_dir / f'split_{split}.json').write_text(json.dumps(rm, indent=2))

print("Rebuilt cflow_datasets + runtime_manifests")

Rebuilt cflow_datasets + runtime_manifests


In [26]:
# 6) Materialize MVTec-like CFLOW datasets per split (main/val) with caching
import os, json, shutil
from pathlib import Path
from PIL import Image

build_root = OUT_ROOT / 'cflow_datasets'
build_root.mkdir(parents=True, exist_ok=True)


def to_png(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    img = Image.open(src).convert('RGB')
    img.save(dst, format='PNG')


def make_mask_like(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    img = Image.open(src)
    m = Image.new('L', img.size, color=255)
    m.save(dst, format='PNG')


def build_mvtec_layout(dst_root, normal_train, normal_test, known_def, unknown_def):
    if dst_root.exists():
        shutil.rmtree(dst_root)

    # train good
    for i, p in enumerate(normal_train):
        to_png(p, dst_root / 'train' / 'good' / f'{i:06d}.png')

    # test good
    for i, p in enumerate(normal_test):
        to_png(p, dst_root / 'test' / 'good' / f'{i:06d}.png')

    # test known defect
    for i, d in enumerate(known_def):
        src = d['path']
        stem = f'{i:06d}'
        img_path = dst_root / 'test' / 'known_defect' / f'{stem}.png'
        to_png(src, img_path)
        make_mask_like(src, dst_root / 'ground_truth' / 'known_defect' / f'{stem}_mask.png')

    # test unknown defect
    for i, d in enumerate(unknown_def):
        src = d['path']
        stem = f'{i:06d}'
        img_path = dst_root / 'test' / 'unknown_defect' / f'{stem}.png'
        to_png(src, img_path)
        make_mask_like(src, dst_root / 'ground_truth' / 'unknown_defect' / f'{stem}_mask.png')


runtime_manifest_dir = OUT_ROOT / 'runtime_manifests'
runtime_manifest_dir.mkdir(parents=True, exist_ok=True)

for split in SPLITS:
    m = json.loads((manifest_dir / f'split_{split}.json').read_text())
    split_root = build_root / f'split_{split}'
    split_root.mkdir(parents=True, exist_ok=True)

    main_root = split_root / 'main'
    val_root = split_root / 'val'

    build_mvtec_layout(
        main_root,
        normal_train=m['normal_train'],
        normal_test=m['normal_test'],
        known_def=m['known_test'],
        unknown_def=m['unknown_test'],
    )
    build_mvtec_layout(
        val_root,
        normal_train=m['normal_train'],
        normal_test=m['normal_val'],
        known_def=m['known_val'],
        unknown_def=m['unknown_val'],
    )

    rm = {
        'split': f'split_{split}',
        'main_counts': {
            'normal': len(m['normal_test']),
            'known': len(m['known_test']),
            'unknown': len(m['unknown_test']),
        },
        'val_counts': {
            'normal': len(m['normal_val']),
            'known': len(m['known_val']),
            'unknown': len(m['unknown_val']),
        },
        'known_test_labels': [x['label'] for x in m['known_test']],
    }
    (runtime_manifest_dir / f'split_{split}.json').write_text(json.dumps(rm, indent=2))

print('Built MVTec-like datasets at', build_root)


Built MVTec-like datasets at /content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/cflow_datasets


In [27]:
"""import os, shutil
from pathlib import Path

CFLOW_REPO = Path('/content/cflow-ad')
OUT_ROOT = Path('/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full')

# pick split you are testing
SPLIT = 'a'
src = OUT_ROOT / 'cflow_datasets' / f'split_{SPLIT}' / 'main'
dst = CFLOW_REPO / 'data' / 'MVTec-AD' / 'bottle'

print("SRC exists:", src.exists(), src)
print("DST parent exists:", dst.parent.exists(), dst.parent)

if not src.exists():
    raise RuntimeError(f"Missing source dataset: {src} (rerun notebook cell thatbuilds cflow_datasets)")

# hard replace runtime dataset
if dst.exists():
    shutil.rmtree(dst)
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, dst)

# verify exact paths CFLOW expects
print("bottle exists:", dst.exists())
print("train exists:", (dst / 'train').exists())
print("train/good count:", len(list((dst / 'train' / 'good').glob('*.png'))))
print("test/good count:", len(list((dst / 'test' / 'good').glob('*.png'))))
print("known_defect count:", len(list((dst / 'test' /
'known_defect').glob('*.png'))))
print("unknown_defect count:", len(list((dst / 'test' /
'unknown_defect').glob('*.png'))))"""

'import os, shutil\nfrom pathlib import Path\n\nCFLOW_REPO = Path(\'/content/cflow-ad\')\nOUT_ROOT = Path(\'/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full\')\n\n# pick split you are testing\nSPLIT = \'a\'\nsrc = OUT_ROOT / \'cflow_datasets\' / f\'split_{SPLIT}\' / \'main\'\ndst = CFLOW_REPO / \'data\' / \'MVTec-AD\' / \'bottle\'\n\nprint("SRC exists:", src.exists(), src)\nprint("DST parent exists:", dst.parent.exists(), dst.parent)\n\nif not src.exists():\n    raise RuntimeError(f"Missing source dataset: {src} (rerun notebook cell thatbuilds cflow_datasets)")\n\n# hard replace runtime dataset\nif dst.exists():\n    shutil.rmtree(dst)\ndst.parent.mkdir(parents=True, exist_ok=True)\nshutil.copytree(src, dst)\n\n# verify exact paths CFLOW expects\nprint("bottle exists:", dst.exists())\nprint("train exists:", (dst / \'train\').exists())\nprint("train/good count:", len(list((dst / \'train\' / \'good\').glob(\'*.png\'))))\nprint("test/good count:", len(list((dst / \'test\

In [28]:
from pathlib import Path
import subprocess, sys

train_py = Path('/content/cflow-ad/train.py')
txt = train_py.read_text()

if "os.makedirs('./results', exist_ok=True)" not in txt:
    txt = txt.replace(
        "        det_roc_auc = roc_auc_score(gt_label, score_label)\n",
        "        det_roc_auc = roc_auc_score(gt_label, score_label)\n"
        "        os.makedirs('./results', exist_ok=True)\n",
        1
    )
    train_py.write_text(txt)

subprocess.check_call([sys.executable, "-m", "py_compile", str(train_py)])
print("Patched train.py: ensure ./results exists before np.savez")

Patched train.py: ensure ./results exists before np.savez


In [29]:
import subprocess
p = subprocess.run([
    "python","main.py",
    "--dataset","mvtec","--class-name","bottle",
    "--action-type","norm-train","--run-name","5000",
    "--batch-size","8","--meta-epochs","1","--sub-epochs","1",
    "--workers","2","--input-size","256","--gpu","0",
], cwd="/content/cflow-ad", capture_output=True, text=True)
print("returncode:", p.returncode)
print("\nSTDOUT:\n", p.stdout[-4000:])
print("\nSTDERR:\n", p.stderr[-4000:])

returncode: 1

STDOUT:
 LR schedule: [0, 0, 0]
Number of pool layers = 3
CNF coder: 512
CNF coder: 1024
CNF coder: 2048


STDERR:
 /usr/local/lib/python3.12/dist-packages/FrEIA/modules/all_in_one_block.py:119: UserWarning: Soft permutation will take a very long time to initialize with 1024 feature channels. Consider using hard permutation instead.
  warnings.warn(("Soft permutation will take a very long time to initialize "
/usr/local/lib/python3.12/dist-packages/FrEIA/modules/all_in_one_block.py:119: UserWarning: Soft permutation will take a very long time to initialize with 2048 feature channels. Consider using hard permutation instead.
  warnings.warn(("Soft permutation will take a very long time to initialize "
Traceback (most recent call last):
  File "/content/cflow-ad/main.py", line 90, in <module>
    main(c)
  File "/content/cflow-ad/main.py", line 83, in main
    train(c)
  File "/content/cflow-ad/train.py", line 277, in train
    train_dataset = MVTecDataset(c, is_train=True

In [30]:
# 7) Run stage-1 CFLOW for all splits (main + val) with artifact reuse
import os, json, shutil, subprocess
from pathlib import Path
import numpy as np

mvtec_root = CFLOW_REPO / 'data' / 'MVTec-AD'
mvtec_root.mkdir(parents=True, exist_ok=True)
runtime_target = mvtec_root / CFLOW_RUNTIME_CLASS

stage1_dir = OUT_ROOT / 'stage1_scores'
stage1_dir.mkdir(parents=True, exist_ok=True)

def latest_score_npz(run_id):
    cands = sorted((CFLOW_REPO / 'results').glob(f'scores_run{run_id}_{CFLOW_RUNTIME_CLASS}_epoch*.npz'))
    return cands[-1] if cands else None

def run_cflow(cmd, cwd):
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if p.returncode != 0:
        print("STDOUT:\n", p.stdout[-5000:])
        print("STDERR:\n", p.stderr[-5000:])
        raise RuntimeError(f"CFLOW failed: {' '.join(cmd)}")
    return p

def swap_in_copy(src_root):
    backup = None
    if runtime_target.exists():
        backup = mvtec_root / f'{CFLOW_RUNTIME_CLASS}_backup'
        if backup.exists():
            shutil.rmtree(backup)
        os.rename(runtime_target, backup)
    shutil.copytree(src_root, runtime_target)
    return backup

def swap_out_copy(backup):
    if runtime_target.exists():
        shutil.rmtree(runtime_target)
    if backup is not None and backup.exists():
        os.rename(backup, runtime_target)

for sidx, split in enumerate(SPLITS):
    print(f"\n===== Stage1 {split} =====")
    rt = json.loads((runtime_manifest_dir / f'split_{split}.json').read_text())
    split_root = build_root / f'split_{split}'

    run_main = 5000 + sidx
    run_val  = 6000 + sidx

    out_main = stage1_dir / f'split_{split}_main.npz'
    out_val  = stage1_dir / f'split_{split}_val.npz'

    if out_main.exists() and out_val.exists():
        print("Reuse existing stage1 score files for split", split)
        continue

    # MAIN
    bkp = swap_in_copy(split_root / 'main')
    try:
        cmd = [
            'python','main.py','--dataset','mvtec','--class-name',CFLOW_RUNTIME_CLASS,
            '--action-type','norm-train','--run-name',str(run_main),
            '--batch-size','8','--meta-epochs','1' if PILOT_MODE else '8',
            '--sub-epochs','1' if PILOT_MODE else '4',
            '--workers','2','--input-size','256','--gpu','0'
        ]
        run_cflow(cmd, str(CFLOW_REPO))
        npz_main = latest_score_npz(run_main)
        if npz_main is None:
            print("results dir files:", sorted(str(x.name) for x in (CFLOW_REPO/'results').glob('*'))[-20:])
            raise RuntimeError(f"No score npz found for split {split} run {run_main}")

        z = np.load(npz_main)
        n, k, u = rt['main_counts']['normal'], rt['main_counts']['known'], rt['main_counts']['unknown']
        score = z['score_label'].reshape(-1)
        if len(score) != (n+k+u):
            raise RuntimeError(f"split {split} main score len mismatch: {len(score)} vs {n+k+u}")
        np.savez(out_main,
                  normal_scores=score[:n].astype('float32'),
                  known_scores=score[n:n+k].astype('float32'),
                  unknown_scores=score[n+k:n+k+u].astype('float32'))
    finally:
        swap_out_copy(bkp)

    # VAL
    bkp = swap_in_copy(split_root / 'val')
    try:
        cmd = [
            'python','main.py','--dataset','mvtec','--class-name',CFLOW_RUNTIME_CLASS,
            '--action-type','norm-train','--run-name',str(run_val),
            '--batch-size','8','--meta-epochs','1','--sub-epochs','1',
            '--workers','2','--input-size','256','--gpu','0'
        ]
        run_cflow(cmd, str(CFLOW_REPO))
        npz_val = latest_score_npz(run_val)
        if npz_val is None:
            print("results dir files:", sorted(str(x.name) for x in (CFLOW_REPO/'results').glob('*'))[-20:])
            raise RuntimeError(f"No score npz found for split {split} run {run_val}")

        z = np.load(npz_val)
        n, k, u = rt['val_counts']['normal'], rt['val_counts']['known'], rt['val_counts']['unknown']
        score = z['score_label'].reshape(-1)
        if len(score) != (n+k+u):
            raise RuntimeError(f"split {split} val score len mismatch: {len(score)} vs {n+k+u}")
        np.savez(out_val,
                  normal_scores=score[:n].astype('float32'),
                  known_scores=score[n:n+k].astype('float32'),
                  unknown_scores=score[n+k:n+k+u].astype('float32'))
    finally:
        swap_out_copy(bkp)

    print("Saved:", out_main, out_val)


===== Stage1 a =====
Reuse existing stage1 score files for split a

===== Stage1 b =====
Reuse existing stage1 score files for split b

===== Stage1 c =====
Reuse existing stage1 score files for split c

===== Stage1 d =====
Reuse existing stage1 score files for split d


In [31]:
import subprocess

cmd = [
    'python', 'main.py',
    '--dataset', 'mvtec',
    '--class-name', 'bottle',
    '--action-type', 'norm-train',
    '--run-name', '5000',
    '--batch-size', '8',
    '--meta-epochs', '1',
    '--sub-epochs', '1',
    '--workers', '2',
    '--input-size', '256',
    '--gpu', '0'
]
p = subprocess.run(cmd, cwd='/content/cflow-ad', capture_output=True, text=True)
print("returncode:", p.returncode)
print("\nSTDOUT:\n", p.stdout[-5000:])
print("\nSTDERR:\n", p.stderr[-5000:])

returncode: 1

STDOUT:
 LR schedule: [0, 0, 0]
Number of pool layers = 3
CNF coder: 512
CNF coder: 1024
CNF coder: 2048


STDERR:
 /usr/local/lib/python3.12/dist-packages/FrEIA/modules/all_in_one_block.py:119: UserWarning: Soft permutation will take a very long time to initialize with 1024 feature channels. Consider using hard permutation instead.
  warnings.warn(("Soft permutation will take a very long time to initialize "
/usr/local/lib/python3.12/dist-packages/FrEIA/modules/all_in_one_block.py:119: UserWarning: Soft permutation will take a very long time to initialize with 2048 feature channels. Consider using hard permutation instead.
  warnings.warn(("Soft permutation will take a very long time to initialize "
Traceback (most recent call last):
  File "/content/cflow-ad/main.py", line 90, in <module>
    main(c)
  File "/content/cflow-ad/main.py", line 83, in main
    train(c)
  File "/content/cflow-ad/train.py", line 277, in train
    train_dataset = MVTecDataset(c, is_train=True

In [32]:
# 8) Train/reuse Stage-2 ResNet50 per split (known-class classifier)
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from pathlib import Path
from PIL import Image

from data import ImageLabelDataset
from src.models.resnet50 import build_resnet50

device = 'cuda' if torch.cuda.is_available() else 'cpu'

TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

stage2_dir = OUT_ROOT / 'stage2'
stage2_dir.mkdir(parents=True, exist_ok=True)


def infer_logits(model, items, class_to_idx):
    ds = ImageLabelDataset([(x['path'], x['label']) for x in items], class_to_idx=class_to_idx, transform=TF)
    dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    logits = []
    labels = []
    model.eval()
    with torch.no_grad():
        for x, y in dl:
            x = x.to(device)
            out = model(x).cpu().numpy()
            logits.append(out)
            labels.append(y.numpy())
    return np.concatenate(logits, axis=0), np.concatenate(labels, axis=0)

for split in SPLITS:
    m = json.loads((manifest_dir / f'split_{split}.json').read_text())
    model_path = stage2_dir / f'split_{split}_resnet50.pt'
    cache_path = stage2_dir / f'split_{split}_logits.npz'

    known_classes = m['known_classes']
    class_to_idx = {c: i for i, c in enumerate(known_classes)}

    if not model_path.exists():
        train_ds = ImageLabelDataset([(x['path'], x['label']) for x in m['known_train']], class_to_idx=class_to_idx, transform=TF)
        val_ds = ImageLabelDataset([(x['path'], x['label']) for x in m['known_val']], class_to_idx=class_to_idx, transform=TF)
        tr = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
        va = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

        model = build_resnet50(num_classes=len(class_to_idx), pretrained=True).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()

        epochs = 2 if PILOT_MODE else 8
        best_acc = -1.0
        best_state = None
        for ep in range(epochs):
            model.train()
            for x, y in tr:
                x, y = x.to(device), y.to(device)
                opt.zero_grad()
                loss = crit(model(x), y)
                loss.backward()
                opt.step()

            model.eval()
            c = 0; n = 0
            with torch.no_grad():
                for x, y in va:
                    x, y = x.to(device), y.to(device)
                    pred = model(x).argmax(1)
                    c += (pred == y).sum().item(); n += y.size(0)
            acc = c / max(1, n)
            if acc > best_acc:
                best_acc = acc
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        torch.save({'state_dict': model.state_dict(), 'known_classes': known_classes}, model_path)
        print(f'split_{split}: stage2 model saved, best_val_acc={best_acc:.4f}')
    else:
        print(f'split_{split}: reusing stage2 model')

    if cache_path.exists():
        print(f'split_{split}: reusing stage2 logits cache')
        continue

    ck = torch.load(model_path, map_location='cpu')
    model = build_resnet50(num_classes=len(known_classes), pretrained=False)
    model.load_state_dict(ck['state_dict'])
    model = model.to(device).eval()

    log_kval, y_kval = infer_logits(model, m['known_val'], class_to_idx)
    log_ktest, y_ktest = infer_logits(model, m['known_test'], class_to_idx)

    # unknown has labels not in class_to_idx; run with dummy idx map
    unk_items = [(x['path'], known_classes[0]) for x in m['unknown_test']]
    from data import ImageLabelDataset as ILD
    unk_ds = ILD(unk_items, class_to_idx={known_classes[0]: 0}, transform=TF)
    unk_dl = DataLoader(unk_ds, batch_size=32, shuffle=False, num_workers=2)
    log_utest = []
    with torch.no_grad():
        for x, _ in unk_dl:
            x = x.to(device)
            log_utest.append(model(x).cpu().numpy())
    log_utest = np.concatenate(log_utest, axis=0)

    np.savez(cache_path,
        logits_known_val=log_kval.astype('float32'),
        labels_known_val=y_kval.astype('int64'),
        logits_known_test=log_ktest.astype('float32'),
        labels_known_test=y_ktest.astype('int64'),
        logits_unknown_test=log_utest.astype('float32')
    )
    print(f'split_{split}: stage2 logits cached')


split_a: reusing stage2 model
split_a: reusing stage2 logits cache
split_b: reusing stage2 model
split_b: reusing stage2 logits cache
split_c: reusing stage2 model
split_c: reusing stage2 logits cache
split_d: reusing stage2 model
split_d: reusing stage2 logits cache


In [33]:
# 9) Final protocol evaluation (per split x FPR), save CSVs, display tables
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

rows = []

for split in SPLITS:
    m = json.loads((manifest_dir / f'split_{split}.json').read_text())
    rt = json.loads((runtime_manifest_dir / f'split_{split}.json').read_text())

    st1_main = np.load(stage1_dir / f'split_{split}_main.npz')
    st1_val = np.load(stage1_dir / f'split_{split}_val.npz')

    s_n_test = st1_main['normal_scores']
    s_k_test = st1_main['known_scores']
    s_u_test = st1_main['unknown_scores']

    # strict validation-only tau calibration (normal-val only)
    s_n_val = st1_val['normal_scores']

    st2 = np.load(stage2_dir / f'split_{split}_logits.npz')
    lk_val = st2['logits_known_val']
    lk_test = st2['logits_known_test']
    lu_test = st2['logits_unknown_test']
    yk_test = st2['labels_known_test']

    def softmax(x):
        x = x - x.max(axis=1, keepdims=True)
        ex = np.exp(x)
        return ex / ex.sum(axis=1, keepdims=True)

    pk_val = softmax(lk_val)
    pk_test = softmax(lk_test)
    pu_test = softmax(lu_test)

    conf_k_val = pk_val.max(axis=1)
    conf_k_test = pk_test.max(axis=1)
    conf_u_test = pu_test.max(axis=1)

    pred_k_test = pk_test.argmax(axis=1)

    kappa = float(np.quantile(conf_k_val, STAGE2_UNKNOWN_Q))

    y_true = np.concatenate([
        np.zeros_like(s_n_test, dtype=np.int64),
        np.ones_like(s_k_test, dtype=np.int64),
        np.ones_like(s_u_test, dtype=np.int64),
    ])
    y_score = np.concatenate([s_n_test, s_k_test, s_u_test])
    auroc = float(roc_auc_score(y_true, y_score))

    for fpr in FPR_GRID:
        tau = float(np.quantile(s_n_val, 1.0 - fpr))

        n_def = s_n_test > tau
        k_def = s_k_test > tau
        u_def = s_u_test > tau

        # two-stage decisions
        k_unknown = conf_k_test < kappa
        u_unknown = conf_u_test < kappa

        # known success: stage1 flags defect AND stage2 assigns correct known class (not unknown)
        k_success = k_def & (~k_unknown) & (pred_k_test == yk_test)

        # unknown success: stage1 flags defect AND stage2 rejects as unknown
        u_success = u_def & u_unknown

        row = {
            'split': f'split_{split}',
            'fpr_target': float(fpr),
            'AUROC_defect_screening': auroc,
            'TPR_defect@FPR': float(np.mean(np.concatenate([k_def, u_def]))),
            'TPR_unknown@FPR': float(np.mean(u_def)),
            'FPR_known_as_def@FPR': float(np.mean(k_def)),
            'FPR_normal_realized': float(np.mean(n_def)),
            'two_stage_known_success': float(np.mean(k_success)),
            'two_stage_unknown_success': float(np.mean(u_success)),
            'stage2_unknown_rate_known_all': float(np.mean(k_unknown)),
            'stage2_unknown_rate_unknown_all': float(np.mean(u_unknown)),
            'tau_stage1': tau,
            'kappa_stage2': kappa,
        }
        rows.append(row)

summary_df = pd.DataFrame(rows).sort_values(['split', 'fpr_target']).reset_index(drop=True)

metric_cols = [
    'AUROC_defect_screening', 'TPR_defect@FPR', 'TPR_unknown@FPR', 'FPR_known_as_def@FPR',
    'FPR_normal_realized', 'two_stage_known_success', 'two_stage_unknown_success',
    'stage2_unknown_rate_known_all', 'stage2_unknown_rate_unknown_all'
]

mean_std = summary_df.groupby('fpr_target', as_index=False)[metric_cols].agg(['mean', 'std'])
mean_std.columns = ['fpr_target'] + [f'{m}_{s}' for m, s in mean_std.columns.tolist()[1:]]

summary_csv = OUT_ROOT / 'cflow_two_stage_summary.csv'
mean_std_csv = OUT_ROOT / 'cflow_two_stage_mean_std.csv'
summary_df.to_csv(summary_csv, index=False)
mean_std.to_csv(mean_std_csv, index=False)

print('Per-split table:')
display(summary_df)
print('Mean/std table:')
display(mean_std)

print('\nSaved CSVs:')
print(summary_csv)
print(mean_std_csv)
print('Summary rows:', len(summary_df))
print('Mean/std rows:', len(mean_std))
print('FPR grid includes 0.15:', 0.15 in sorted(summary_df['fpr_target'].unique().tolist()))


Exception ignored in: <function NpzFile.__del__ at 0x7d69a814c540>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/npyio.py", line 226, in __del__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/npyio.py", line 221, in close
    self.fid.close()
OSError: [Errno 107] Transport endpoint is not connected
Exception ignored in: <function NpzFile.__del__ at 0x7d69a814c540>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/npyio.py", line 226, in __del__
    self.close()
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/npyio.py", line 221, in close
    self.fid.close()
OSError: [Errno 107] Transport endpoint is not connected
Exception ignored in: <function NpzFile.__del__ at 0x7d69a814c540>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/npyio.py", line 226, in __del__
    self.close()
  File "/usr/local/lib/python3.12/dist-pa

Per-split table:


,split,fpr_target,AUROC_defect_screening,TPR_defect@FPR,TPR_unknown@FPR,FPR_known_as_def@FPR,FPR_normal_realized,two_stage_known_success,two_stage_unknown_success,stage2_unknown_rate_known_all,stage2_unknown_rate_unknown_all,tau_stage1,kappa_stage2
0,split_a,0.05,0.567083,0.133333,0.211111,0.055556,0.0500,0.044444,0.000000,0.066667,0.000000,2.023377,0.532364
1,split_a,0.10,0.567083,0.138889,0.222222,0.055556,0.0625,0.044444,0.000000,0.066667,0.000000,2.008058,0.532364
2,split_a,0.15,0.567083,0.294444,0.522222,0.066667,0.1500,0.055556,0.000000,0.066667,0.000000,1.916362,0.532364
3,split_a,0.20,0.567083,0.344444,0.588889,0.100000,0.2000,0.055556,0.000000,0.066667,0.000000,1.865812,0.532364
4,split_b,0.05,0.489167,0.266667,0.233333,0.300000,0.2625,0.266667,0.011111,0.166667,0.133333,1.924000,0.772760
5,split_b,0.10,0.489167,0.311111,0.288889,0.333333,0.4000,0.277778,0.011111,0.166667,0.133333,1.834307,0.772760
6,split_b,0.15,0.489167,0.344444,0.311111,0.377778,0.4250,0.300000,0.011111,0.166667,0.133333,1.815134,0.772760
7,split_b,0.20,0.489167,0.377778,0.355556,0.400000,0.4375,0.311111,0.011111,0.166667,0.133333,1.798478,0.772760
8,split_c,0.05,0.394514,0.050000,0.011111,0.088889,0.0625,0.077778,0.000000,0.111111,0.088889,2.031942,0.579292
9,split_c,0.10,0.394514,0.061111,0.011111,0.111111,0.0875,0.100000,0.000000,0.111111,0.088889,1.981722,0.579292


Mean/std table:


,fpr_target,AUROC_defect_screening_mean,AUROC_defect_screening_std,TPR_defect@FPR_mean,TPR_defect@FPR_std,TPR_unknown@FPR_mean,TPR_unknown@FPR_std,FPR_known_as_def@FPR_mean,FPR_known_as_def@FPR_std,FPR_normal_realized_mean,FPR_normal_realized_std,two_stage_known_success_mean,two_stage_known_success_std,two_stage_unknown_success_mean,two_stage_unknown_success_std,stage2_unknown_rate_known_all_mean,stage2_unknown_rate_known_all_std,stage2_unknown_rate_unknown_all_mean,stage2_unknown_rate_unknown_all_std
0,0.05,0.482969,0.070572,0.134722,0.094322,0.119444,0.119110,0.150000,0.108298,0.103125,0.106739,0.130556,0.097868,0.002778,0.005556,0.125,0.0457,0.097222,0.072222
1,0.10,0.482969,0.070572,0.162500,0.105641,0.138889,0.137736,0.186111,0.126157,0.168750,0.156292,0.155556,0.103836,0.005556,0.006415,0.125,0.0457,0.097222,0.072222
2,0.15,0.482969,0.070572,0.223611,0.121325,0.236111,0.228499,0.211111,0.138778,0.228125,0.131646,0.175000,0.107869,0.008333,0.010638,0.125,0.0457,0.097222,0.072222
3,0.20,0.482969,0.070572,0.258333,0.127778,0.266667,0.258995,0.250000,0.132249,0.262500,0.117704,0.194444,0.112765,0.011111,0.015713,0.125,0.0457,0.097222,0.072222



Saved CSVs:
/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/cflow_two_stage_summary.csv
/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/cflow_two_stage_mean_std.csv
Summary rows: 16
Mean/std rows: 4
FPR grid includes 0.15: True


In [34]:
from pathlib import Path
base = Path('/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full')
print((base/'cflow_two_stage_summary.csv').exists(), base/'cflow_two_stage_summary.csv')
print((base/'cflow_two_stage_mean_std.csv').exists(), base/'cflow_two_stage_mean_std.csv')

True /content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/cflow_two_stage_summary.csv
True /content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/cflow_two_stage_mean_std.csv


## Run Order
1. Run cells `0` to `10` top-to-bottom.
2. Keep `PILOT_MODE=True` first to validate end-to-end quickly.
3. Set `PILOT_MODE=False` and rerun from cell `4` onward for heavier run.
4. Final required files are:
   - `/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/cflow_two_stage_summary.csv`
   - `/content/drive/MyDrive/fyp_outputs/severstral_cflow_two_stage_full/cflow_two_stage_mean_std.csv`